# Loading Data from Json files to Delta Tabes

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Configurations

In [0]:
config = {
    "resources":["Patient", "Encounter", "Observation", "Condition"],
    "raw_path":"/Volumes/fhir_data/prd_raw/firstv/raw",
    "bronze_path": "fhir_data.prd_bronze",
    "base_url": "https://hapi.fhir.org/baseR4"
}


#dbutils.fs.rm('/Volumes/project/project_schema/firstv/bronze/condition',True)

run_date = datetime.now().strftime("%Y-%m-%d")

df = spark.read.format('json').load(f'/Volumes/fhir_data/prd_raw/firstv/raw/encounter/date={run_date}/response_{run_date}.json')
df.display()

# Writing data to Bronze delta

In [0]:
from pyspark.sql.functions import explode, col, lit, to_json
from pyspark.sql.types import StructType, ArrayType
from datetime import datetime

def load_bronze(resource, run_date, config):

    raw_file = f"{config['raw_path']}/{resource.lower()}/date={run_date}/response_{run_date}.json"
    print(f"Loading {resource} from {raw_file}")

    # Read raw JSON
    raw_df = spark.read.format("json").option("multiline", "true").load(raw_file)
    raw_df.printSchema()
    #raw_df.display()

    df = raw_df

    #Check schema of entries safely
    if "entries" in df.columns:
        entries_field = df.schema["entries"].dataType

        # Case 1: Correct FHIR format (array<struct>)
        if isinstance(entries_field, ArrayType) and isinstance(entries_field.elementType, StructType):
            df = df.withColumn("entry", explode("entries")).select("entry.*")

        # Case 2: Bad format (array<string>) → skip
        else:
            print(f"Skipping {resource} due to unexpected schema: {entries_field}")
            return 0
    else:
        print(f"'entries' column not found in {resource}")
        return 0

    # Convert nested resource to string (important for bronze)
    if "resource" in df.columns:
        df = df.withColumn("resource", to_json(col("resource")))

    # Add metadata columns
    df = df.withColumns({
        "extraction_timestamp": lit(datetime.now()),
        "api_url": lit(config['base_url']),
        "load_date": lit(run_date),
        "source_file": col("_metadata.file_path")
    })

    record_count = df.count()
    print(f"Total records for {resource} to write : {record_count}")

    # Write to Bronze
    bronze_tb = f"{config['bronze_path']}.{resource.lower()}"

    df.write.mode("append") \
        .option("mergeSchema", "true") \
        .format("delta") \
        .saveAsTable(bronze_tb)

    print(f"Written data to Bronze : {bronze_tb}")

    return record_count

# Main loading all resources into Bronze

In [0]:
run_date = datetime.now().strftime("%Y-%m-%d")
print(f"Run date : {run_date}")
print('=' * 50)
bronze_log = []

for resource in config['resources']:
  print(f"\nProcessing the Data for :{resource}")
  print('=' * 50)
  try:
    count = load_bronze(resource,run_date,config)
    bronze_log.append({
      "resource":resource,
      "run_date":run_date,
      "records":count,
      "status":"Success"
    })
  except Exception as e:
    print(f"Error while processing {resource} : {e}")
    bronze_log.append({
      "resource":resource,
      "run_date":run_date,
      "records":0,
      "status": f"Failed: {e}"
    })
print("\n" + "=" * 50)
print("Bronze oad Summary")
print("=" * 50)
for log in bronze_log:
  print(f"Resource : {log['resource']}, Records : {log['records']}, Status : {log['status']}")


# checking the Bronze Tables

In [0]:
for resource in config['resources']:
  print(f"\n{resource} Bronze preview")
  bronze_path = f"{config['bronze_path']}.{resource.lower()}"
  try:
    df = spark.read.table(bronze_path)
    print(f"Total records for {resource} : {df.count()}")
    print(f" Columns : {df.columns[:6]}") # checking first 6 columns")
    df.select('load_date','extraction_timestamp','api_url','source_file').show(3,False)

  except Exception as e:
    print(f"Could Not read bronze table for {resource} :  {e}")

# checking delta table history

In [0]:
from delta.tables import DeltaTable

for resource in config['resources']:
  bronze_tb = f"{config['bronze_path']}.{resource.lower()}"
  try:
    dt = DeltaTable.forName(spark,bronze_tb)
    print(f"\n{resource} Delta History:")
    dt.history(5).select('version','timestamp','operation','operationParameters').orderBy(col('version').desc()).show(truncate=False)

  except Exception as e:
    print(f"Could Not read bronze table for {resource} :  {e}")